# Open-ended achievements: three planner designs on one budget

The chain notebooks pose one recipe in a carved world. Here the world is an
unmodified Craftax-Classic map and the objective is *"Unlock as many
achievements as you can."* Three designs run on the same skills, executor,
worlds and budget:

1. `saycan_recipe_aware`: SayCan with the recipe-aware VLM prompt as the Say
   term and the competence prior as the Can term.
2. `subgoal_vlm`: the VLM proposes formal sub-goals over the achievements
   its skills unlock, and the symbolic planner follows them, re-asking on low vitals, nearby
   threats and nightfall. When every answer is rejected and no earlier
   sub-goal is still reachable, it takes the curriculum's next sub-goal
   instead of ending the episode; those calls are counted apart.
3. `subgoal_scripted`: the same sub-goal planner with a fixed tech-tree
   curriculum (`ScriptedGoalSelector`) in place of the VLM - what the VLM
   has to beat.

Each episode gets 80 skill calls and 3000 environment steps, and ends early
on death or when the planner has nothing left to propose. A skill hands
control back to its planner early when a zombie comes within three tiles or
the player is hit, since the planners decide only between skills. Every
design runs with creatures off and on, on matched seeds. Scores follow Crafter:
per-achievement success rates and their geometric mean. The planners offer
every skill the executor was trained on, and each skill unlocks one
achievement, so the skill set caps the score. The seven-skill executor
reaches seven achievements (collect_wood, place_table, make_wood_pickaxe,
make_wood_sword, collect_stone, make_stone_pickaxe, collect_drink); the
sixteen-skill executor adds make_stone_sword, collect_coal, place_stone,
place_furnace, collect_sapling, place_plant, eat_cow, wake_up and
eat_plant, and the eighteen-skill executor adds defeat_zombie and
defeat_skeleton, leaving only the iron and diamond achievements out of
reach. The designs differ in how reliably and how cheaply they reach the
cap, and in how they survive.

Start it in a fresh kernel so the JAX memory fraction and the Hugging Face
cache location are set before JAX and Transformers initialize.

In [ ]:
# Install Tiny-SIMA and the optional VLM backend into this kernel.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Leave room for the PyTorch VLM before importing JAX.
os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.35"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

# Keep downloaded VLM weights beside the repository, on the same volume,
# rather than wherever the image points HF_HOME; TINY_SIMA_HF_HOME overrides
# the location. Hugging Face reads it once, on first import.
os.environ["HF_HOME"] = os.environ.get(
    "TINY_SIMA_HF_HOME", str(repo_root.parent / "hf-cache")
)

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,vlm,cuda12" if has_cuda else "notebook,vlm"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

# Import Torchvision before Transformers. Transformers caches optional
# dependency availability on first import.
torch = importlib.import_module("torch")
torchvision = importlib.import_module("torchvision")
transformers = importlib.import_module("transformers")
outlines = importlib.import_module("outlines")
transformers_utils = importlib.import_module("transformers.utils")
if not transformers_utils.is_torchvision_available():
    raise RuntimeError(
        "Transformers cached Torchvision as unavailable. Restart this "
        "kernel, then run this installation cell before any other cell."
    )
huggingface_constants = importlib.import_module("huggingface_hub.constants")
if Path(huggingface_constants.HF_HOME) != Path(os.environ["HF_HOME"]):
    raise RuntimeError(
        "Hugging Face was imported before HF_HOME was set. Restart this "
        "kernel, then run this installation cell before any other cell."
    )

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("CUDA detected:", has_cuda)
print("JAX memory fraction:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])
print("Hugging Face cache:", huggingface_constants.HF_HUB_CACHE)

In [ ]:
import dataclasses
import json
import time

import jax

from tiny_sima.config import load_vlm_planner_config
from tiny_sima.envs import make_open_ended_environment
from tiny_sima.evaluation import (
    ACHIEVEMENT_NAMES,
    ACHIEVEMENT_TIERS,
    DEFAULT_INTERRUPTION,
    DEFAULT_MAX_ENVIRONMENT_STEPS,
    DEFAULT_MAX_SKILL_CALLS,
    WOOD_TO_STONE_PICKAXE_CHAIN,
    collect_affordance_samples,
    evaluate_open_ended,
    fingerprint,
    fit_affordance_competence_prior,
    git_commit,
    json_ready_open_ended_result,
    load_goal_conditioned_checkpoint,
    make_learned_skill_executor,
    open_ended_planners,
    provenance_mismatch,
    run_provenance,
    with_provenance,
)
from tiny_sima.goals import ALL_SKILL_IDS, STONE_PICKAXE_SKILL_IDS
from tiny_sima.memory import EpisodeMemory
from tiny_sima.planner import (
    OBSERVATION_PRESETS,
    CompetencePrior,
    TransformersVLMBackend,
    make_critic_affordance_scorer,
)

print("JAX devices:", jax.devices())

In [ ]:
# The executor notebook 07 selected under this preset. "sixteen_skill" adds
# the survival skills and the rest of the stone stage, and "eighteen_skill"
# the combat skills; "stone_pickaxe" is the seven-skill executor of the
# earlier open-ended runs. The planners offer every skill the executor knows,
# so it must know at least those seven.
EXECUTOR_PRESET = "eighteen_skill"
EXECUTOR_SELECTION_PATH = (
    repo_root
    / "runs"
    / f"executor-improvement-{EXECUTOR_PRESET}"
    / "selected_executor.json"
)
if not EXECUTOR_SELECTION_PATH.is_file():
    raise FileNotFoundError(
        f"{EXECUTOR_SELECTION_PATH} does not exist. Train and select the "
        f"{EXECUTOR_PRESET} preset in notebook 07, or set EXECUTOR_PRESET to "
        '"stone_pickaxe".'
    )
EXECUTOR_CHECKPOINT = (
    repo_root / json.loads(EXECUTOR_SELECTION_PATH.read_text())["checkpoint"]
)
# The 8B scaling test; vlm_planner_qwen3_2b.yaml for the 2B baseline. The
# VLM configuration is part of each result's provenance, so a saved design
# is never reused across model sizes.
VLM_CONFIG_PATH = repo_root / "configs" / "vlm_planner_qwen3_8b.yaml"

# The competence prior probes every skill the executor knows at each boundary
# of the stone-pickaxe chain, whose boundaries cover the first seven skills.
# A later skill is fitted on the boundaries where its precondition holds, such
# as collect_coal once the wood pickaxe is made; one the chain never enables,
# such as eat_plant, keeps the shrunk base rate. The seven-skill prior is
# shared with notebook 06 when it has already been saved there.
PRIOR_CHAIN = WOOD_TO_STONE_PICKAXE_CHAIN
REFIT_PRIOR = False
NUM_CALIBRATION_EPISODES = 12
CALIBRATION_SEED = 7

# What the VLM sees besides the frame. The open-ended prompts lean on vitals,
# creatures and unlocked achievements, which the image alone does not state;
# set to None for the image alone, as in notebook 06.
OBSERVATION = OBSERVATION_PRESETS["image_and_state"]

# 16 episodes, as in the seed-51 runs; every episode decodes in lockstep with
# the others, but a sub-goal design can ask the VLM up to three times per
# decision, so lower it for a quick check.
NUM_EPISODES = 16
# Seed 51 is the development seed; 202 confirms its results on fresh worlds.
# Run the notebook once per seed: 51, then again with 202.
EVALUATION_SEED = 51
SKILL_HORIZON = 64
MAX_SKILL_CALLS = DEFAULT_MAX_SKILL_CALLS  # 80
MAX_ENVIRONMENT_STEPS = DEFAULT_MAX_ENVIRONMENT_STEPS  # 3000
# A skill ends early, handing back to the planner, when a zombie comes within
# three tiles or the player is hit; None runs every skill to its horizon.
INTERRUPTION = DEFAULT_INTERRUPTION
# The sub-goal designs are offered a night shelter (planner/shelter.py):
# walling the player in with stone to sleep until morning. The VLM chooses
# whether to take it now, after its sub-goals (such as drinking first) or
# not at all, and whether to fight a close zombie; the curriculum shelters
# whenever it can and always fights. With creatures off there is nothing to
# hide from or fight, so those runs leave both off and stay reusable.
SHELTER_AT_NIGHT = True
MOB_SETTINGS = (False, True)
DESIGNS = ("saycan_recipe_aware", "subgoal_vlm", "subgoal_scripted")
# (design, mobs) runs reloaded from the newest saved run of this seed instead
# of rerun, for a change that cannot affect them. A saved result is reused
# only when its provenance -- executor, VLM configuration, observation, prior,
# episodes, seed and limits -- matches this run's, and is otherwise rerun;
# provenance cannot see code, so rerun a design whose own code changed.
# Skill interruption is part of a creatures-on run's provenance, and the
# night shelter of a creatures-on sub-goal run's, so every design reruns with
# creatures on. With creatures off nothing interrupts a
# skill, so those runs are reused, except seed 202's sub-goal runs, which
# predate the survival change (zombie defence, alerts at 5, the curriculum's
# food rule, skeleton memory). A new executor changes the provenance, so
# everything reruns with it.
REUSE_RUNS = (
    ("saycan_recipe_aware", False),
    *(
        ((design, False) for design in ("subgoal_vlm", "subgoal_scripted"))
        if EVALUATION_SEED == 51
        else ()
    ),
)

RUNS_DIR = repo_root / "runs" / "open-ended"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

vlm_config = load_vlm_planner_config(VLM_CONFIG_PATH)
print("VLM:", vlm_config.model_id)
print("Executor:", EXECUTOR_CHECKPOINT, "exists:", EXECUTOR_CHECKPOINT.exists())
print(f"Episodes: {NUM_EPISODES} per design and setting, seed {EVALUATION_SEED}")

In [ ]:
backend = TransformersVLMBackend(vlm_config)

checkpoint = load_goal_conditioned_checkpoint(EXECUTOR_CHECKPOINT)
executor = make_learned_skill_executor(checkpoint, deterministic=False)
missing = [
    skill for skill in STONE_PICKAXE_SKILL_IDS if skill not in executor.skill_ids
]
if missing:
    raise ValueError(
        f"The executor was not trained on {[skill.value for skill in missing]}. "
        "Train a seven-skill executor in notebook 07 and select it."
    )
# Every skill the executor knows, in catalog order.
SKILLS = tuple(skill for skill in ALL_SKILL_IDS if skill in executor.skill_ids)
print(f"Skills offered ({len(SKILLS)}):", [skill.value for skill in SKILLS])

# Notebook 06 saves its seven-skill prior under the plain name, so a larger
# catalog gets its own file.
prior_suffix = "" if SKILLS == STONE_PICKAXE_SKILL_IDS else f"_{len(SKILLS)}_skills"
PRIOR_PATH = (
    EXECUTOR_CHECKPOINT.parents[1]
    / f"competence_prior_{PRIOR_CHAIN.name}{prior_suffix}.json"
)

if PRIOR_PATH.exists() and not REFIT_PRIOR:
    competence_prior = CompetencePrior.load(PRIOR_PATH)
    print("Loaded competence prior:", PRIOR_PATH)
else:
    started = time.perf_counter()
    fit_samples = collect_affordance_samples(
        make_critic_affordance_scorer(checkpoint),
        executor,
        chain=PRIOR_CHAIN,
        num_episodes=NUM_CALIBRATION_EPISODES,
        seed=CALIBRATION_SEED,
        candidate_skills=SKILLS,
        skill_horizon=SKILL_HORIZON,
    )
    competence_prior = fit_affordance_competence_prior(fit_samples, skills=SKILLS)
    competence_prior.save(PRIOR_PATH)
    print(
        f"Fitted competence prior on {len(fit_samples)} probes in "
        f"{time.perf_counter() - started:.1f}s: {PRIOR_PATH}"
    )

## Closed loop

Each (design, mobs) pair runs the same `NUM_EPISODES` keys, so the worlds
match across designs, and the two mob settings share their maps (the
creatures are the only difference). Planners are rebuilt per setting so no
response cache crosses settings.

In [ ]:
def design_provenance(design, mobs):
    """Everything a design's outcome depends on, besides code."""

    return run_provenance(
        executor=EXECUTOR_CHECKPOINT,
        vlm=dataclasses.asdict(vlm_config),
        observation=None if OBSERVATION is None else repr(OBSERVATION),
        competence_prior=fingerprint(competence_prior.to_dict()),
        num_episodes=NUM_EPISODES,
        seed=EVALUATION_SEED,
        skill_horizon=SKILL_HORIZON,
        max_skill_calls=MAX_SKILL_CALLS,
        max_environment_steps=MAX_ENVIRONMENT_STEPS,
        mobs=mobs,
        # Without creatures nothing interrupts a skill, so the rule is left
        # out and runs from before it still match.
        interruption=(
            INTERRUPTION.to_dict() if mobs and INTERRUPTION is not None else None
        ),
        # Likewise the night shelter and the fight choice, which only the
        # sub-goal designs' goal source makes: now or after its sub-goals,
        # unlike runs where it could only shelter now and every close zombie
        # was fought.
        shelter=(
            "goal_source_choice_now_or_after"
            if mobs and SHELTER_AT_NIGHT and design.startswith("subgoal")
            else None
        ),
        fight=(
            "goal_source_choice" if mobs and design.startswith("subgoal") else None
        ),
    )


def saved_design(design, mobs):
    """Return the newest reusable saved result, or None and why there is none."""

    provenance = design_provenance(design, mobs)
    reason = "not saved"
    for run in sorted(RUNS_DIR.glob(f"seed{EVALUATION_SEED}-*"), reverse=True):
        path = run / f"{design}_mobs-{'on' if mobs else 'off'}.json"
        if not path.is_file():
            continue
        result = json.loads(path.read_text())
        mismatch = provenance_mismatch(result, provenance)
        if mismatch is None:
            return result, None
        if reason == "not saved":
            reason = f"{mismatch} since {run.name}"
    return None, reason


commit = git_commit(repo_root)
print("commit:", commit)
results = {}
# The commit each reused (design, mobs) result ran at.
reused = {}
for mobs in MOB_SETTINGS:
    environment = make_open_ended_environment(mobs=mobs)
    planners = open_ended_planners(
        SKILLS,
        backend,
        vlm_config,
        competence_prior,
        OBSERVATION,
        shelter_at_night=SHELTER_AT_NIGHT and mobs,
        choose_fights=mobs,
    )
    for design in DESIGNS:
        if (design, mobs) in REUSE_RUNS:
            saved, mismatch = saved_design(design, mobs)
            if saved is not None:
                results[(design, mobs)] = saved
                reused[(design, mobs)] = saved["commit"]
                summary = saved["summary"]
                print(
                    f"{design:22s} mobs={mobs!s:5s} "
                    f"score={summary['crafter_score']:6.2f} "
                    f"achievements={summary['achievements_per_episode']['mean']:.2f}"
                    f" reused={saved['commit']}"
                )
                continue
            print(f"{design:22s} mobs={mobs!s:5s} rerunning: {mismatch}")
        started = time.perf_counter()
        result = evaluate_open_ended(
            planners[design],
            executor,
            environment,
            num_episodes=NUM_EPISODES,
            seed=EVALUATION_SEED,
            skill_horizon=SKILL_HORIZON,
            max_skill_calls=MAX_SKILL_CALLS,
            max_environment_steps=MAX_ENVIRONMENT_STEPS,
            memory_factory=EpisodeMemory,
            interruption=INTERRUPTION,
        )
        result["design"] = design
        result["seconds"] = time.perf_counter() - started
        results[(design, mobs)] = with_provenance(
            result, design_provenance(design, mobs), commit
        )
        summary = result["summary"]
        print(
            f"{design:22s} mobs={mobs!s:5s} "
            f"score={summary['crafter_score']:6.2f} "
            f"achievements={summary['achievements_per_episode']['mean']:.2f} "
            f"seconds={result['seconds']:.0f}"
        )

## Metrics

Achievements per episode are mean and standard deviation over episodes.
The Crafter score is `exp(mean_i ln(1 + s_i)) - 1` over all 22
achievements with `s_i` in percent, so an achievement no episode unlocks
pulls it down as much as any other. VLM calls are goal-source calls: the
scripted selector's answers are counted the same way, so its column shows
how often the sub-goal planner re-asks.

In [ ]:
header = (
    f"{'design':22s} {'mobs':5s} {'score':>6s} {'ach mean':>8s} {'ach std':>7s} "
    f"{'skills':>6s} {'steps':>6s} {'vlm':>5s} {'fallback':>8s}  terminations"
)
print(header)
print("-" * len(header))
for (design, mobs), result in results.items():
    summary = result["summary"]
    # Results saved before the fallback existed have no fallback calls.
    fallback = summary.get("fallback_calls", {"mean": 0.0})["mean"]
    terminations = {
        name: count for name, count in summary["termination_counts"].items() if count
    }
    print(
        f"{design:22s} {mobs!s:5s} {summary['crafter_score']:6.2f} "
        f"{summary['achievements_per_episode']['mean']:8.2f} "
        f"{summary['achievements_per_episode']['std']:7.2f} "
        f"{summary['skill_calls']['mean']:6.1f} "
        f"{summary['environment_steps']['mean']:6.0f} "
        f"{summary['vlm_calls']['mean']:5.1f} "
        f"{fallback:8.1f}  {terminations}"
    )

# How often the goal source leaned on the fallback, and where its own
# shelter and fight answers differ from the curriculum's rule in the same
# state: shelter whenever it could start, always fight. Results saved
# before these were recorded show n/a.
print()
print("design                 mobs   fallback  answers unlike the curriculum")
for (design, mobs), result in results.items():
    summary = result["summary"]
    share = summary.get("fallback_share")
    parts = []
    for kind, stats in summary.get("choices", {}).items():
        if stats["answers"]:
            parts.append(
                f"{kind} {stats['differ']}/{stats['answers']} {stats['counts']}"
            )
    print(
        f"{design:22s} {mobs!s:5s} "
        f"{'n/a' if share is None else f'{share:.0%}':>8s}  "
        f"{'; '.join(parts) or 'n/a'}"
    )

In [ ]:
# Per-achievement success rates, in tier order; achievements no run unlocked
# are left out to keep the table short.
columns = list(results)
header = " ".join(f"{d[:12]:>12s}/{'on' if m else 'off':3s}" for d, m in columns)
print(f"{'achievement':22s} {header}")
for names in ACHIEVEMENT_TIERS.values():
    for name in names:
        rates = [results[key]["summary"]["success_rates"][name] for key in columns]
        if any(rates):
            print(f"{name:22s} " + " ".join(f"{rate:16.2f}" for rate in rates))

In [ ]:
# Tier breakdown: the mean success rate of each tier's achievements, and how
# many of them an episode unlocks.
print(f"{'tier':14s} {header}")
for tier in ACHIEVEMENT_TIERS:
    cells = []
    for key in columns:
        tier_summary = results[key]["summary"]["tiers"][tier]
        cells.append(
            f"{tier_summary['success_rate']:7.2f} "
            f"({tier_summary['achievements_per_episode']['mean']:.1f})".rjust(16)
        )
    print(f"{tier:14s} " + " ".join(cells))

In [ ]:
# One JSON file per run, beside a manifest of what was run.
stamp = time.strftime("%Y%m%d-%H%M%S")
run_dir = RUNS_DIR / f"seed{EVALUATION_SEED}-{stamp}"
run_dir.mkdir(parents=True, exist_ok=True)
for (design, mobs), result in results.items():
    path = run_dir / f"{design}_mobs-{'on' if mobs else 'off'}.json"
    path.write_text(json.dumps(json_ready_open_ended_result(result), indent=2))
manifest = {
    "commit": commit,
    # Designs reloaded from an earlier run, by the commit each ran at.
    "reused": {
        f"{design}_mobs-{'on' if mobs else 'off'}": reused_commit
        for (design, mobs), reused_commit in reused.items()
    },
    "executor": str(EXECUTOR_CHECKPOINT),
    "vlm": vlm_config.model_id,
    "observation": None if OBSERVATION is None else repr(OBSERVATION),
    "competence_prior": str(PRIOR_PATH),
    "num_episodes": NUM_EPISODES,
    "seed": EVALUATION_SEED,
    "skill_horizon": SKILL_HORIZON,
    "max_skill_calls": MAX_SKILL_CALLS,
    "max_environment_steps": MAX_ENVIRONMENT_STEPS,
    "interruption": None if INTERRUPTION is None else INTERRUPTION.to_dict(),
    "shelter_at_night": SHELTER_AT_NIGHT,
    "achievements": list(ACHIEVEMENT_NAMES),
    "tiers": {tier: list(names) for tier, names in ACHIEVEMENT_TIERS.items()},
    "summaries": {
        f"{design}_mobs-{'on' if mobs else 'off'}": result["summary"]
        for (design, mobs), result in results.items()
    },
}
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print("Saved:", run_dir)

## Fights, staged

Ten of the thirteen creatures-on deaths came inside `defeat_zombie`. A
closed-loop trace cannot tell a fight lost to the odds from one lost to the
executor, so here the fight is staged: a creature at distance 1-3 in an
open arena, the player at health 2-9 with no, a wood or a stone sword, one
creature or two, by day or by night, 50 rollouts per cell, for
`defeat_zombie`, `defeat_skeleton` and `eat_cow`. The executor runs
stochastically, as in the closed loop; a scripted chase-and-hit reference
runs on the same worlds and environment keys. Only cells 1-4 above need to
have run.

*Adjacent* steps start with a target creature one step away, *facing* ones
with it on the tile the player faces. A *hit* is DO while facing, a *miss*
DO while not, a *retreat* a move while adjacent. A creature beside the
player with its cooldown spent strikes before the player's second action,
so the reference's losses mark the cells nobody wins; the gap to the
reference is the executor's.

In [ ]:
from tiny_sima.evaluation import (
    FIGHT_BENCHMARK_SEED,
    benchmark_fights,
    summarize_fights,
)

FIGHT_ROLLOUTS_PER_CELL = 50
FIGHT_CELL_FIELDS = ("distance", "health", "sword", "mobs", "night")

started = time.perf_counter()
fight_rows = benchmark_fights(
    executor,
    rollouts_per_cell=FIGHT_ROLLOUTS_PER_CELL,
    seed=FIGHT_BENCHMARK_SEED,
    skill_horizon=SKILL_HORIZON,
)
print(f"{len(fight_rows)} staged fights in {time.perf_counter() - started:.0f}s")

# Grouped by health and sword, the table a fight feasibility term reads.
fight_tables = {
    "summary": summarize_fights(fight_rows),
    "by_health_sword": summarize_fights(
        fight_rows, by=("skill_id", "policy", "health", "sword")
    ),
    "cells": summarize_fights(
        fight_rows, by=("skill_id", "policy", *FIGHT_CELL_FIELDS)
    ),
}
fight_dir = RUNS_DIR / f"fights-{time.strftime('%Y%m%d-%H%M%S')}"
fight_dir.mkdir(parents=True, exist_ok=True)
(fight_dir / "fights.json").write_text(
    json.dumps(
        {
            "commit": git_commit(repo_root),
            "executor": str(EXECUTOR_CHECKPOINT),
            "seed": FIGHT_BENCHMARK_SEED,
            "rollouts_per_cell": FIGHT_ROLLOUTS_PER_CELL,
            "skill_horizon": SKILL_HORIZON,
            **fight_tables,
        },
        indent=2,
    )
)
print("Saved:", fight_dir)

In [ ]:
def rate(value):
    return "    -" if value is None else f"{value:5.2f}"


header = (
    f"{'skill':16s} {'policy':8s} {'win':>5s} {'died':>5s} {'win steps':>9s} "
    f"{'win cost':>8s} {'adjacent':>8s} {'facing':>6s} {'hits':>5s} "
    f"{'misses':>6s} {'retreats':>8s} {'1st hit':>7s}  outcomes"
)
print(header)
print("-" * len(header))
for row in fight_tables["summary"]:
    print(
        f"{row['skill_id']:16s} {row['policy']:8s} {rate(row['win_rate'])} "
        f"{rate(row['death_rate'])} {rate(row['win_steps']):>9s} "
        f"{rate(row['win_health_lost']):>8s} {rate(row['adjacent_share']):>8s} "
        f"{rate(row['facing_when_adjacent']):>6s} {rate(row['hits'])} "
        f"{rate(row['misses']):>6s} {rate(row['retreats']):>8s} "
        f"{rate(row['first_hit_step']):>7s}  {row['outcomes']}"
    )

# Win and death rates by start health and sword, executor against the
# reference: where the executor falls short of what the fight allows.
by_health_sword = {
    (row["skill_id"], row["policy"], row["health"], row["sword"]): row
    for row in fight_tables["by_health_sword"]
}
for skill in dict.fromkeys(row["skill_id"] for row in fight_rows):
    swords = list(dict.fromkeys(row["sword"] for row in fight_rows))
    print(f"\n{skill}: win (died), executor | chase")
    print(f"{'health':>6s} " + " ".join(f"{sword:>25s}" for sword in swords))
    for health in sorted({row["health"] for row in fight_rows}):
        cells = []
        for sword in swords:
            sides = [
                by_health_sword[(skill, policy, health, sword)]
                for policy in ("executor", "chase")
            ]
            cells.append(
                " | ".join(
                    f"{side['win_rate']:.2f} ({side['death_rate']:.2f})"
                    for side in sides
                ).rjust(25)
            )
        print(f"{health:6d} " + " ".join(cells))

# The other settings, one at a time.
for field in ("distance", "mobs", "night"):
    print(f"\nby {field}: win (died), executor | chase")
    split = {
        (row["skill_id"], row["policy"], row[field]): row
        for row in summarize_fights(fight_rows, by=("skill_id", "policy", field))
    }
    for skill, _, value in sorted(key for key in split if key[1] == "executor"):
        sides = [split[(skill, policy, value)] for policy in ("executor", "chase")]
        print(
            f"  {skill:16s} {field}={value!s:5s} "
            + " | ".join(
                f"{side['win_rate']:.2f} ({side['death_rate']:.2f})" for side in sides
            )
        )